# VishGuard — Colab demo (T5.2)

Launches the Streamlit UI on Colab T4 and exposes it publicly via an
ngrok tunnel. Use this notebook to record the Phase 5 demo video.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Add a Colab secret named `NGROK_AUTH_TOKEN` with your free ngrok
   token from [dashboard.ngrok.com](https://dashboard.ngrok.com/authtokens).
3. Set `VISHGUARD_REPO_URL` below (or set it as a Colab secret).

**Demo script (1–2 min):**
- 0:00 — narrate the vishing problem
- 0:15 — show Streamlit UI, upload a WAV
- 0:30 — transcript + spoof score + tactics populate
- 0:50 — risk score card + spoken briefing plays
- 1:10 — mention one known failure (see docs/FAILURES.md)
- 1:25 — show GitHub URL + close

In [ ]:
# Cell 1 — repo setup (self-contained; identical pattern to other notebooks)
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = os.environ.get('VISHGUARD_REPO_URL', 'https://github.com/ben-blake/vishguard.git')
DRIVE_SRC = '/content/drive/MyDrive/vishguard'
REPO_DIR  = Path('/content/vishguard')
ON_COLAB  = 'google.colab' in sys.modules or Path('/content').exists()

def sh(cmd, check=True):
    print('$', cmd)
    subprocess.run(cmd, shell=True, check=check)

if ON_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print('Drive mount skipped:', e)

    if 'PLACEHOLDER' not in REPO_URL:
        if REPO_DIR.exists():
            sh(f'cd {REPO_DIR} && git pull --ff-only')
        else:
            sh(f'git clone {REPO_URL} {REPO_DIR}')
    elif Path(DRIVE_SRC).exists():
        REPO_DIR.mkdir(parents=True, exist_ok=True)
        sh(f'rsync -a --delete --exclude .venv --exclude __pycache__ {DRIVE_SRC}/ {REPO_DIR}/')
    else:
        raise RuntimeError('Set VISHGUARD_REPO_URL or copy the repo to MyDrive/vishguard/')

    sh(f'pip install -q -e {REPO_DIR}')
    sh(f'pip install -q -r {REPO_DIR}/requirements.txt')
    try:
        import torch
        if torch.cuda.is_available():
            sh(f'pip install -q -r {REPO_DIR}/requirements-gpu.txt')
    except Exception as e:
        print('GPU extras skipped:', e)

    src = str(REPO_DIR / 'src')
    if src not in sys.path:
        sys.path.insert(0, src)

import vishguard
print('vishguard', vishguard.__version__)

In [ ]:
# Cell 2 — install pyngrok and configure auth token
import subprocess
subprocess.run(['pip', 'install', '-q', 'pyngrok'], check=True)

import os
from pyngrok import ngrok, conf

# Pull token from Colab secrets (preferred) or env var
try:
    from google.colab import userdata
    ngrok_token = userdata.get('NGROK_AUTH_TOKEN')
except Exception:
    ngrok_token = os.environ.get('NGROK_AUTH_TOKEN', '')

if not ngrok_token:
    raise RuntimeError(
        'Set NGROK_AUTH_TOKEN in Colab secrets (Secrets panel on the left) '
        'or as an environment variable. Free account at dashboard.ngrok.com.'
    )

conf.get_default().auth_token = ngrok_token
print('ngrok auth token configured')

In [ ]:
# Cell 3 — launch Streamlit in background, open ngrok tunnel
import subprocess, time
from pathlib import Path
from pyngrok import ngrok

REPO_DIR = Path('/content/vishguard')
PORT = 8501

# Start Streamlit (headless so it doesn't try to open a browser)
_streamlit_proc = subprocess.Popen(
    [
        'streamlit', 'run',
        str(REPO_DIR / 'app.py'),
        f'--server.port={PORT}',
        '--server.headless=true',
        '--server.enableCORS=false',
        '--server.enableXsrfProtection=false',
    ],
    cwd=str(REPO_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Give Streamlit a few seconds to start before tunneling
time.sleep(5)

tunnel = ngrok.connect(PORT, 'http')
public_url = tunnel.public_url

print('=' * 60)
print('VishGuard Streamlit UI:')
print(public_url)
print('=' * 60)
print('Open the URL above in a new tab to use the app.')
print('Run Cell 4 to shut down when done.')

In [ ]:
# Cell 4 (OPTIONAL) — CLI smoke test without the UI
# Useful for verifying the full pipeline works end-to-end before recording.
# Downloads a short LibriSpeech clip and runs the CLI.
import subprocess, json
from pathlib import Path

REPO_DIR  = Path('/content/vishguard')
AUDIO_URL = 'https://www.openslr.org/resources/12/dev-clean.tar.gz'
SAMPLE    = Path('/tmp/sample_test.wav')
OUT_DIR   = Path('/tmp/vishguard_out')
OUT_DIR.mkdir(exist_ok=True)

if not SAMPLE.exists():
    # Download a single short clip via datasets (avoids unpacking the full tarball)
    from datasets import load_dataset
    ds = load_dataset('librispeech_asr', 'clean', split='validation', streaming=True)
    item = next(iter(ds))
    import soundfile as sf
    sf.write(str(SAMPLE), item['audio']['array'], item['audio']['sampling_rate'])
    print(f'Saved sample: {SAMPLE} ({len(item["audio"]["array"])/item["audio"]["sampling_rate"]:.1f}s)')

# Run the pipeline CLI
result = subprocess.run(
    ['python', '-m', 'vishguard.cli', '--audio', str(SAMPLE), '--out-dir', str(OUT_DIR)],
    cwd=str(REPO_DIR),
    capture_output=True,
    text=True,
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

# Pretty-print the report JSON
report_files = list(OUT_DIR.glob('*.json'))
if report_files:
    report = json.loads(report_files[-1].read_text())
    print(json.dumps(report, indent=2))

In [ ]:
# Cell 5 — shutdown (run after recording to free port + kill tunnel)
from pyngrok import ngrok

ngrok.kill()

try:
    _streamlit_proc.terminate()
    _streamlit_proc.wait(timeout=5)
    print('Streamlit stopped')
except Exception as e:
    print('Shutdown:', e)